# **Patient Feedback Data Cleaning**

## 1. Importing Libraries and Loading Data

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

import sys
sys.path.append(str(Path.cwd().parent))

from scripts.paths import RAW_DIR, CLEANED_DIR

df = pd.read_csv(RAW_DIR / "12_patient_feedback_dirty.csv")

## 2. Initial Data Inspection

In [2]:
df.head()

,Feedback_ID,Patient_ID,Encounter_ID,Feedback_Date,Cleanliness_Rating,Doctor_Rating,Nurse_Rating,Wait_Time_Rating,Overall_Rating,Complaint_Type
0,F900000,P101520,E204317,2025-05-19,3.0,NaN,2.0,4.0,2.0,Billing
1,F900001,P107721,E203301,2025-07-21,1.0,5.0,1.0,4.0,5.0,Cleanliness
2,F900002,P102203,E206686,2026-08-23,NaN,1.0,5.0,4.0,5.0,Food
3,F900003,P102280,E200227,2025-04-09,2.0,1.0,3.0,4.0,5.0,Cleanliness
4,F900004,P102746,E208431,2025-08-19,NaN,4.0,2.0,4.0,5.0,NaN


In [3]:
print("Shape:", df.shape)

print("\nColumn names:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nMissing values:")
print(df.isna().sum())

print("\nDuplicate rows:", df.duplicated().sum())

print("\nBasic statistics:")
display(df.describe(include="all"))

Shape: (7000, 10)

Column names:
['Feedback_ID', 'Patient_ID', 'Encounter_ID', 'Feedback_Date', 'Cleanliness_Rating', 'Doctor_Rating', 'Nurse_Rating', 'Wait_Time_Rating', 'Overall_Rating', 'Complaint_Type']

Data types:
Feedback_ID            object
Patient_ID             object
Encounter_ID           object
Feedback_Date          object
Cleanliness_Rating    float64
Doctor_Rating         float64
Nurse_Rating          float64
Wait_Time_Rating      float64
Overall_Rating        float64
Complaint_Type         object
dtype: object

Missing values:
Feedback_ID              0
Patient_ID               0
Encounter_ID             0
Feedback_Date            0
Cleanliness_Rating     793
Doctor_Rating         1560
Nurse_Rating           884
Wait_Time_Rating      1129
Overall_Rating         883
Complaint_Type        1708
dtype: int64

Duplicate rows: 0

Basic statistics:


,Feedback_ID,Patient_ID,Encounter_ID,Feedback_Date,Cleanliness_Rating,Doctor_Rating,Nurse_Rating,Wait_Time_Rating,Overall_Rating,Complaint_Type
count,7000,7000,7000,7000,6207.000000,5440.000000,6116.000000,5871.000000,6117.000000,5292
unique,7000,4636,5584,601,NaN,NaN,NaN,NaN,NaN,6
top,F900000,P106941,E213193,2025-06-15,NaN,NaN,NaN,NaN,NaN,Cleanliness
freq,1,7,4,23,NaN,NaN,NaN,NaN,NaN,905
mean,NaN,NaN,NaN,NaN,3.757048,3.426838,3.472368,2.972066,3.423410,NaN
std,NaN,NaN,NaN,NaN,1.478366,1.417128,1.392206,1.412612,1.391919,NaN
min,NaN,NaN,NaN,NaN,1.000000,1.000000,1.000000,1.000000,1.000000,NaN
25%,NaN,NaN,NaN,NaN,3.000000,2.000000,2.000000,2.000000,2.000000,NaN
50%,NaN,NaN,NaN,NaN,5.000000,4.000000,4.000000,3.000000,4.000000,NaN
75%,NaN,NaN,NaN,NaN,5.000000,5.000000,5.000000,4.000000,5.000000,NaN


## 3. Audit of Specific Columns

In [4]:
# ID format and uniqueness
for column, pattern in [
    ("Feedback_ID", r"F\d{6}"),
    ("Patient_ID", r"P\d{6}"),
    ("Encounter_ID", r"E\d{6}")
]:
    values = df[column].astype("string")
    invalid_count = (
        values.notna()
        & ~values.str.fullmatch(pattern, na=False)
    ).sum()

    print(f"{column} - duplicate values:", df[column].duplicated().sum())
    print(f"{column} - invalid format:", invalid_count)

# Rating columns
rating_columns = [
    "Cleanliness_Rating",
    "Doctor_Rating",
    "Nurse_Rating",
    "Wait_Time_Rating",
    "Overall_Rating"
]

for column in rating_columns:
    s = df[column]

    print(f"\n{column}")
    print("Missing:", s.isna().sum())
    print("Minimum:", s.min())
    print("Maximum:", s.max())
    print("Non-integer values:", (s.dropna() % 1 != 0).sum())
    print(
        "Values outside 1-5:",
        ((s.notna()) & ((s < 1) | (s > 5))).sum()
    )
    print("Unique values:", sorted(s.dropna().unique()))

# Complaint type
print("\nComplaint_Type values:")
print(df["Complaint_Type"].value_counts(dropna=False))

print(
    "Complaint_Type whitespace issues:",
    (
        df["Complaint_Type"].astype("string")
        != df["Complaint_Type"].astype("string").str.strip()
    ).sum()
)

# Feedback date
feedback_date_temp = pd.to_datetime(df["Feedback_Date"], errors="coerce")

print("\nFeedback_Date invalid/unparseable:",
      feedback_date_temp.isna().sum())
print("Minimum Feedback_Date:", feedback_date_temp.min())
print("Maximum Feedback_Date:", feedback_date_temp.max())

# Optional cross-table checks against encounters master
encounters_path = CLEANED_DIR / "05_encounters_cleaned.csv"

if encounters_path.exists():
    encounters_df = pd.read_csv(encounters_path)

    encounter_lookup = (
        encounters_df
        .drop_duplicates("Encounter_ID")
        .set_index("Encounter_ID")
    )

    valid_encounter_ids = set(encounter_lookup.index.dropna())

    print(
        "\nEncounter_IDs not found in encounters master:",
        (~df["Encounter_ID"].isin(valid_encounter_ids)).sum()
    )

    expected_patient_id = df["Encounter_ID"].map(
        encounter_lookup["Patient_ID"]
    )

    mismatch_mask = (
        expected_patient_id.notna()
        & (df["Patient_ID"] != expected_patient_id)
    )

    print(
        "Patient_ID / Encounter_ID mismatches:",
        mismatch_mask.sum()
    )
else:
    print(
        "\nEncounters master not found in CLEANED_DIR; "
        "cross-table validation skipped."
    )

Feedback_ID - duplicate values: 0
Feedback_ID - invalid format: 0
Patient_ID - duplicate values: 2364
Patient_ID - invalid format: 0
Encounter_ID - duplicate values: 1416
Encounter_ID - invalid format: 0

Cleanliness_Rating
Missing: 793
Minimum: 1.0
Maximum: 5.0
Non-integer values: 0
Values outside 1-5: 0
Unique values: [1.0, 2.0, 3.0, 4.0, 5.0]

Doctor_Rating
Missing: 1560
Minimum: 1.0
Maximum: 5.0
Non-integer values: 0
Values outside 1-5: 0
Unique values: [1.0, 2.0, 3.0, 4.0, 5.0]

Nurse_Rating
Missing: 884
Minimum: 1.0
Maximum: 5.0
Non-integer values: 0
Values outside 1-5: 0
Unique values: [1.0, 2.0, 3.0, 4.0, 5.0]

Wait_Time_Rating
Missing: 1129
Minimum: 1.0
Maximum: 5.0
Non-integer values: 0
Values outside 1-5: 0
Unique values: [1.0, 2.0, 3.0, 4.0, 5.0]

Overall_Rating
Missing: 883
Minimum: 1.0
Maximum: 5.0
Non-integer values: 0
Values outside 1-5: 0
Unique values: [1.0, 2.0, 3.0, 4.0, 5.0]

Complaint_Type values:
Complaint_Type
NaN                1708
Cleanliness         905
Bill

## 4. Data Cleaning

### 4.1 Feedback_Date

In [5]:
feedback_date_cleaned = pd.to_datetime(
    df["Feedback_Date"],
    errors="coerce"
)

df["Feedback_Date"] = feedback_date_cleaned

### 4.2 Complaint_Type

In [6]:
# Remove accidental leading/trailing whitespace if present.
# Missing Complaint_Type values are retained as missing because they
# may legitimately represent feedback without a complaint.
complaint_cleaned = df["Complaint_Type"].astype("string").str.strip()

df["Complaint_Type"] = complaint_cleaned

### 4.3 Rating Columns

In [7]:
# Rating columns are expected to be whole-number scores from 1 to 5.
# Missing ratings are retained because they represent non-responses;
# they should not be imputed without a documented business rule.

for column in rating_columns:
    df[column] = df[column].astype("Float64")

## 5. Final Validation

In [8]:
print("Final shape:", df.shape)
print("Duplicate rows:", df.duplicated().sum())
print("Duplicate Feedback_ID:", df["Feedback_ID"].duplicated().sum())

# ID format validation
for column, pattern in [
    ("Feedback_ID", r"F\d{6}"),
    ("Patient_ID", r"P\d{6}"),
    ("Encounter_ID", r"E\d{6}")
]:
    values = df[column].astype("string")
    print(
        f"Invalid {column} after cleaning:",
        (
            values.notna()
            & ~values.str.fullmatch(pattern, na=False)
        ).sum()
    )

# Date validation
print(
    "Invalid Feedback_Date after cleaning:",
    pd.to_datetime(df["Feedback_Date"], errors="coerce").isna().sum()
)

# Rating validation
for column in rating_columns:
    s = df[column].dropna()
    print(
        f"{column} values outside 1-5 after cleaning:",
        ((s < 1) | (s > 5)).sum()
    )

# Complaint validation
print(
    "Complaint_Type whitespace issues after cleaning:",
    (
        df["Complaint_Type"].astype("string")
        != df["Complaint_Type"].astype("string").str.strip()
    ).sum()
)

# Optional cross-table validation
encounters_path = CLEANED_DIR / "05_encounters_cleaned.csv"

if encounters_path.exists():
    encounters_df = pd.read_csv(encounters_path)

    encounter_lookup = (
        encounters_df
        .drop_duplicates("Encounter_ID")
        .set_index("Encounter_ID")
    )

    expected_patient_id = df["Encounter_ID"].map(
        encounter_lookup["Patient_ID"]
    )

    print(
        "Encounter_IDs not found in encounters master after cleaning:",
        (~df["Encounter_ID"].isin(encounter_lookup.index)).sum()
    )

    print(
        "Patient_ID / Encounter_ID mismatches after cleaning:",
        (
            expected_patient_id.notna()
            & (df["Patient_ID"] != expected_patient_id)
        ).sum()
    )
else:
    print(
        "Encounters master not found; cross-table validation skipped."
    )

print("\nFinal dtypes:")
print(df.dtypes)

print("\nFinal missing values:")
print(df.isna().sum())

Final shape: (7000, 10)
Duplicate rows: 0
Duplicate Feedback_ID: 0
Invalid Feedback_ID after cleaning: 0
Invalid Patient_ID after cleaning: 0
Invalid Encounter_ID after cleaning: 0
Invalid Feedback_Date after cleaning: 0
Cleanliness_Rating values outside 1-5 after cleaning: 0
Doctor_Rating values outside 1-5 after cleaning: 0
Nurse_Rating values outside 1-5 after cleaning: 0
Wait_Time_Rating values outside 1-5 after cleaning: 0
Overall_Rating values outside 1-5 after cleaning: 0
Complaint_Type whitespace issues after cleaning: 0
Encounter_IDs not found in encounters master after cleaning: 0
Patient_ID / Encounter_ID mismatches after cleaning: 6998

Final dtypes:
Feedback_ID                   object
Patient_ID                    object
Encounter_ID                  object
Feedback_Date         datetime64[ns]
Cleanliness_Rating           Float64
Doctor_Rating                Float64
Nurse_Rating                 Float64
Wait_Time_Rating             Float64
Overall_Rating               Floa

## 6. Export Cleaned Data

In [10]:
df.to_csv(CLEANED_DIR / "12_patient_feedback_cleaned.csv", index=False)